# Spindle — Star Schema Transform

Spindle generates normalized 3NF data and can transform it into a **star schema** (dimension + fact tables) ready for Power BI DirectLake mode.

**What this notebook covers**
- Generating the Retail domain
- Transforming to a star schema with `StarSchemaTransform`
- Exploring dimension and fact tables
- The auto-generated `dim_date` table
- Surrogate keys (`sk_*`) and natural key preservation (`nk_*`)
- Writing the star schema to a Fabric Lakehouse
- Healthcare star schema

In [ ]:
%pip install sqllocks-spindle --quiet

## Generate and transform — Retail

In [ ]:
from sqllocks_spindle import Spindle, RetailDomain
from sqllocks_spindle.transform import StarSchemaTransform

spindle = Spindle()
result = spindle.generate(domain=RetailDomain(), scale="fabric_demo", seed=42)

transform = StarSchemaTransform()
star = transform.transform(result.tables, RetailDomain().star_schema_map())
print(star.summary())

## Dimension tables

In [ ]:
# dim_customer — surrogate key as first column
star.dimensions["dim_customer"].head()

In [ ]:
# dim_product — enriched with product_category columns (prefixed 'cat_')
star.dimensions["dim_product"].head()

In [ ]:
# Category columns added by enrichment
cat_cols = [c for c in star.dimensions["dim_product"].columns if c.startswith("cat_")]
print(f"Enriched category columns: {cat_cols}")

## dim_date — auto-generated

`dim_date` is automatically generated from the date range found in your fact data.
The `sk_date` key uses YYYYMMDD integer format for efficient range filtering.

In [ ]:
star.date_dim.head(10)

In [ ]:
print(f"Date range: {star.date_dim['date'].min()} → {star.date_dim['date'].max()}")
print(f"Total days: {len(star.date_dim):,}")
print(f"Columns: {list(star.date_dim.columns)}")

## Fact tables

Fact tables contain surrogate key (`sk_*`) columns for joining to dimensions, plus preserved natural keys (`nk_*`) for traceability.

In [ ]:
star.facts["fact_sale"].head()

In [ ]:
# sk_date is YYYYMMDD integer — joins directly to dim_date
sale = star.facts["fact_sale"]
print(f"sk_date sample: {sale['sk_date'].dropna().head().tolist()}")
print(f"SK columns:     {[c for c in sale.columns if c.startswith('sk_')]}")
print(f"NK columns:     {[c for c in sale.columns if c.startswith('nk_')]}")

## Referential integrity check

In [ ]:
fact_sale = star.facts["fact_sale"]
dim_customer = star.dimensions["dim_customer"]
dim_product  = star.dimensions["dim_product"]

valid_customers = set(dim_customer["sk_customer"])
valid_products  = set(dim_product["sk_product"])

assert fact_sale["sk_customer"].dropna().isin(valid_customers).all(), "Customer SK mismatch"
assert fact_sale["sk_product"].dropna().isin(valid_products).all(),  "Product SK mismatch"
print("SK referential integrity: PASS")

## Write star schema to Fabric Lakehouse

> **Remove or skip this section when running locally.** Requires `spark` to be available (Fabric notebook).

In [ ]:
# Uncomment to run inside a Fabric notebook
# for table_name, df in star.all_tables().items():
#     spark.createDataFrame(df).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"retail_star_{table_name}")
#     print(f"  retail_star_{table_name:<25} {len(df):>6,} rows")
print("Uncomment the block above to write to your Fabric Lakehouse.")

## Healthcare star schema

In [ ]:
from sqllocks_spindle import HealthcareDomain

hc_result = spindle.generate(domain=HealthcareDomain(), scale="fabric_demo", seed=42)
hc_star = transform.transform(hc_result.tables, HealthcareDomain().star_schema_map())
print(hc_star.summary())

In [ ]:
hc_star.facts["fact_encounter"].head()

## CDM export

Spindle can also export to Microsoft Common Data Model (CDM) format — compatible with Dataverse, Power Platform, and Fabric CDM connectors.

In [ ]:
from sqllocks_spindle.transform import CdmMapper

mapper = CdmMapper()
entity_map = RetailDomain().cdm_map()

# In-memory model.json manifest
model = mapper.to_model_json(result.tables, domain_name="SpindleRetail", entity_map=entity_map)

print(f"CDM model name: {model['name']}")
print(f"Entities: {[e['name'] for e in model['entities']]}")

In [ ]:
# Write CDM folder to disk (local or ADLS path)
import tempfile, os

with tempfile.TemporaryDirectory() as tmpdir:
    files = mapper.write_cdm_folder(
        result.tables,
        output_dir=tmpdir,
        domain_name="SpindleRetail",
        entity_map=entity_map,
    )
    print(f"Written {len(files)} files:")
    for f in files:
        print(f"  {os.path.relpath(f, tmpdir)}")